# 01 — Data understanding and EDA

**Business question:** Can compressor sensors identify unusual operating behavior? MetroPT-3 has no row-wise failure labels; UCI supplies four air-leak incident intervals. This notebook is exploratory and does not modify the source data.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import RAW, PROCESSED, MODELS, ANALOG, DIGITAL, FAILURES

from src.data.load_data import load_raw
from src.evaluation.evaluate import event_mask
df=load_raw()
print('Shape:',df.shape); display(df.head(),df.dtypes.to_frame('dtype'))

## Integrity, time coverage and sampling

UCI reports both 1 Hz and 0.1 Hz in different paragraphs. Empirical timestamp intervals take precedence. Missingness and duplicate timestamps are diagnosed without removal.

In [ ]:
t=pd.to_datetime(df['timestamp'],errors='coerce')
print('Dates:',t.min(),t.max())
print('Nulls:',df.isna().sum().sort_values(ascending=False).head(20))
print('Duplicate rows:',df.duplicated().sum(),'duplicate timestamps:',t.duplicated().sum())
print('Sampling delta:',t.sort_values().diff().value_counts().head(10))
numeric=df.select_dtypes(include='number')
print('Infinite numeric entries:',np.isinf(numeric.to_numpy()).sum())
display(numeric.describe().T)
print('Analog present:',set(ANALOG)&set(df.columns),'Digital present:',set(DIGITAL)&set(df.columns))

## Distributions

Inspect skew, multimodality and implausible spikes. Peaks are not removed: they may reflect physical changes or faults.

In [ ]:
cols=[c for c in ['TP2','TP3','Motor_current','Oil_temperature'] if c in df]
fig,axs=plt.subplots(len(cols),1,figsize=(11,2.7*len(cols)),squeeze=False)
for ax,c in zip(axs.flat,cols): sns.histplot(df[c].dropna(),bins=60,ax=ax);ax.set(title=f'Distribution: {c}',xlabel=c,ylabel='Count')
plt.tight_layout()

## Time series

Downsample to hourly medians for legibility. Averages/medians can hide short transients, so investigate suspicious intervals at original resolution later.

In [ ]:
hourly=df.assign(timestamp=t).set_index('timestamp')[cols].resample('1h').median()
ax=hourly.plot(subplots=True,figsize=(12,8),title='Hourly median sensor readings',legend=False)
for a,c in zip(ax,cols):a.set_ylabel(c)
plt.xlabel('Timestamp');plt.tight_layout()

## Correlations

Pearson correlations describe linear co-movement, not causation; shared operating modes and temporal dependence may inflate apparent relationships.

In [ ]:
corr=numeric[[c for c in ANALOG if c in numeric]].corr()
plt.figure(figsize=(9,6));sns.heatmap(corr,annot=True,cmap='coolwarm',center=0)
plt.title('Analog sensor Pearson correlations');plt.xlabel('Sensor');plt.ylabel('Sensor');plt.tight_layout()

## Documented incidents

Four UCI-reported air-leak periods are *interval-level reference events*, not exhaustive ground truth. Check temporal overlap, but do not manufacture binary failure labels.

In [ ]:
display(pd.DataFrame(FAILURES,columns=['incident_start','incident_end']))
print('Observations within reported intervals:',event_mask(t).sum())

## Decision

Use unsupervised anomaly detection with chronological train/validation/test partitions. Validate timestamp cadence, missingness, and sensor semantics before interpreting alert counts. No conclusions about predictive lead time can be drawn yet.